# Food recommendations from blood tests

This notebook shows how **biofoodrec** works, using the demo patient from the Biolabs website.

You give it blood test results. It tells you:

1. which results are high or low
2. which foods fit this person best, and which to limit
3. which nutrients to eat more of or less of
4. why each food got its place

**How to run it:** click a grey code box and press **Shift + Enter**. Go from top to bottom the first time.
After that you can change any number and run that box again.

## 1. Start

In [1]:
import json

import numpy as np
import pandas as pd
from biofoodrec import FoodRecommender, Tuning, sample_patient

pd.set_option("display.max_rows", 80)
pd.set_option("display.max_colwidth", 70)

BLUE, RED = "#2a78d6", "#e34948"

def bars(table, *columns, axis=0):
    """Small bars inside a table: blue to the right of zero, red to the left."""
    return (table.style.bar(subset=list(columns), axis=axis, align="zero", color=[RED, BLUE], height=60, width=80)
            .format(precision=2))

rec = FoodRecommender(tuning=Tuning.recommended())
ref = rec.reference
print(len(ref.biochemicals), "blood tests |", len(ref.foods), "foods |", len(ref.nutrients), "nutrients")

58 blood tests | 150 foods | 75 nutrients


The Biolabs data is inside the package, so there is nothing to download.
`Tuning.recommended()` measures every test the same way. Section 10 explains why.

## 2. Load a patient from JSON

This is the demo patient the Biolabs website loads: a 46 year old woman with 4 visits.
Here are the first 3 results of her last visit, exactly as they sit in the JSON:

In [2]:
patient = sample_patient()
print("Visits:", len(patient["biometrics_entries"]), "| sex:", patient["user"]["gender"])
patient["biometrics_entries"][-1]["biometrics"][:3]

Visits: 4 | sex: female


[{'id': 1, 'name': 'Blood Glucose', 'unit': 'mg/dL', 'value': 102.4},
 {'id': 2, 'name': 'Hemoglobin A1c', 'unit': '%', 'value': 5.4},
 {'id': 3, 'name': 'Glycated Albumin', 'unit': '%', 'value': 13.5}]

Give the whole JSON to the recommender. It takes the newest result of every test.

In [3]:
result = rec.recommend(patient)
result

<Recommendation: 150 foods, best: Flax Seed, Chia Seed, Parsley>

These JSON shapes work too. You can pass a Python object, a JSON string, or a path to a file:

```text
{"Blood Glucose": 130, "Vitamin D": 14}
{"Blood Glucose": {"value": 130}}
[{"id": 1, "value": 130}]
{"data": {"updatedBiochemicalsData": [...], "unExpiredBiometricsData": [...]}}
{"user": {"gender": "female"}, "biometrics_entries": [...]}
```

The first three do not say the sex, so add `sex="female"` or `sex="male"`, like `rec.recommend(labs, sex="female")`.

## 3. Lab results

**How far off** tells you where a result sits. 0 is the middle of the healthy range. 1 or minus 1 is the edge.
More than 1 means outside the range. Blue bars are above the middle, red bars are below.
Results outside the range come first.

In [4]:
bars(result.labs().head(15), "How far off")

,Result,Healthy range,Status,How far off
Biochemical,,,,
C-Reactive Protein,4 mg/L,0 to 3,High,1.67
Gamma-Glutamyl Transferase,43 U/L,5 to 36,High,1.45
Thyroid-Stimulating Hormone (TSH),4.4 mIU/L,0.4 to 4,High,1.22
Uric Acid,6.3 mg/dL,2.4 to 6,High,1.17
Blood Glucose,102.4 mg/dL,70 to 100,High,1.16
Vitamin D,18.4 ng/mL,20 to 50,Low,-1.11
Iron,47.8 µg/dL,50 to 170,Low,-1.04
Total Cholesterol,198.3 mg/dL,125 to 200,"Normal, upper side",0.95
Progesterone,1.8 ng/mL,0.1 to 25,"Normal, lower side",-0.86


## 4. The short report

One text you can read or send. It names the results that are out of range, the nutrients to eat more and
less of, and the best and worst foods.

In [5]:
print(result.summary())

58 lab results, 7 outside the healthy range.
High: C-Reactive Protein 4 mg/L (healthy 0 to 3); Gamma-Glutamyl Transferase 43 U/L (healthy 5 to 36); Thyroid-Stimulating Hormone (TSH) 4.4 mIU/L (healthy 0.4 to 4); Uric Acid 6.3 mg/dL (healthy 2.4 to 6); Blood Glucose 102.4 mg/dL (healthy 70 to 100)
Low: Vitamin D 18.4 ng/mL (healthy 20 to 50); Iron 47.8 µg/dL (healthy 50 to 170)

Eat more: Omega-3 Fatty Acids, Vitamin C, Antioxidants, Fiber, Magnesium, Polyphenols
Eat less: Trans Fatty Acids, Alcohol, Fats, Oxalates, Carbohydrates, Phytic Acid

Best 10 foods (score out of 100):
  1. Flax Seed  93.9  helps lower Blood Glucose and C-Reactive Protein
  2. Chia Seed  93  helps lower C-Reactive Protein and Blood Glucose
  3. Parsley  87.6  helps lower C-Reactive Protein and Blood Glucose
  4. Mint  86.6  helps lower C-Reactive Protein and Blood Glucose
  5. Salmon  84.5  helps raise Vitamin D and Iron
  6. Thyme  82.2  helps lower Blood Glucose and C-Reactive Protein
  7. Sardines  82.2  help

## 5. Foods

**Score** is out of 100. A food gets points three ways: what it does to each blood test directly, what its
nutrients do, and its general quality (nutriscore). 100 would mean the best on all three.
**Why** names the tests the food helps most, starting with the ones out of range.

In [6]:
result.foods(20)

,Food,Group,Score,Why
Rank,,,,
1,Flax Seed,Seed,93.9,helps lower Blood Glucose and C-Reactive Protein
2,Chia Seed,Seed,93.0,helps lower C-Reactive Protein and Blood Glucose
3,Parsley,Vegetable,87.6,helps lower C-Reactive Protein and Blood Glucose
4,Mint,Vegetable,86.6,helps lower C-Reactive Protein and Blood Glucose
5,Salmon,Seafood,84.5,helps raise Vitamin D and Iron
6,Thyme,Vegetable,82.2,helps lower Blood Glucose and C-Reactive Protein
7,Sardines,Seafood,82.2,helps raise Vitamin D and Iron
8,Strawberry,Fruit,82.1,helps lower C-Reactive Protein and raise Iron
9,Pumpkin Seed,Seed,82.1,helps raise Vitamin D and Iron


Foods to limit, worst first:

In [7]:
result.limit(10)

,Food,Group,Score,Why
Rank,,,,
150,Brownies,Nosh,2.8,raises Uric Acid and Blood Glucose
149,Donuts,Nosh,10.0,raises Blood Glucose and Uric Acid
148,Muffins,Nosh,10.0,raises Uric Acid and Blood Glucose
147,Cakes,Nosh,10.3,raises Blood Glucose and Uric Acid
146,Pastries,Nosh,10.8,raises Uric Acid and Gamma-Glutamyl Transferase
145,Cookies,Nosh,11.6,raises Uric Acid and Gamma-Glutamyl Transferase
144,Croissants,Nosh,12.7,raises Uric Acid and Gamma-Glutamyl Transferase
143,Pies,Nosh,18.6,raises Blood Glucose and Uric Acid
142,Potato Chips,Nosh,19.4,raises Uric Acid and Gamma-Glutamyl Transferase


## 6. Nutrients

**Need** goes from 100 (eat much more) to minus 100 (eat much less).
**Found in** lists the richest foods. For nutrients to eat more of, it only lists foods that also suit this person.

In [8]:
nutrients = result.nutrients()
bars(nutrients.head(12), "Need")

,Advice,Need,Group,Found in
Nutrient,,,,
Omega-3 Fatty Acids,Eat more,86,Fatty Acids,"Flax Seed, Chia Seed, Pine Nut"
Vitamin C,Eat more,73,Vitamins,"Guava, Thyme, Parsley"
Antioxidants,Eat more,70,Other,"Mint, Thyme, Cauliflower"
Fiber,Eat more,67,Other,"Oregano, Chia Seed, Flax Seed"
Magnesium,Eat more,38,Minerals,"Pumpkin Seed, Flax Seed, Sesame Seed"
Polyphenols,Eat more,38,Phytochemicals,"Guacamole, Shiitake Mushrooms, Portobello Mushrooms"
Polyunsaturated Fatty Acids,Eat more,29,Fatty Acids,"Pine Nut, Flax Seed, Chia Seed"
Flavonoids,Eat more,27,Phytochemicals,"Oregano, Mint, Thyme"
Monounsaturated Fatty Acids,Eat more,26,Fatty Acids,"Hazelnut, Almond, Walnut"


Nutrients to eat less of, strongest first:

In [9]:
bars(nutrients[nutrients.Advice == "Eat less"].sort_values("Need").head(10), "Need")

,Advice,Need,Group,Found in
Nutrient,,,,
Trans Fatty Acids,Eat less,-100,Fatty Acids,"Ghee, Butter, Pepperoni"
Alcohol,Eat less,-72,Other,Beer
Fats,Eat less,-60,Macronutrients,"Ghee, Butter, Mayonnaise"
Oxalates,Eat less,-53,Other,"Celery, Cassava, Beets"
Carbohydrates,Eat less,-53,Macronutrients,"Cereals, Popcorn, Barley"
Phytic Acid,Eat less,-51,Other,"Crackers, Yams, Potatoes"
Fructose,Eat less,-51,Sugars,"Brownies, Banana, Cakes"
Saturated Fatty Acids,Eat less,-51,Fatty Acids,"Ghee, Butter, Cheddar"
Caffeine,Eat less,-50,Other,"Tea, Coffee, Coca Cola"


## 7. Why a food got its place

Each row is one blood test.

1. **Your result**: high, low or normal
2. **This food**: does the food raise or lower that test
3. **Direct** and **Through nutrients**: the two ways the food acts
4. **Total**: positive means this test pushes the food up the list, negative pushes it down

Change the food name and run the box again.

In [10]:
bars(result.explain("Flax Seed").head(12), "Total")

,Your result,This food,Direct,Through nutrients,Total,Verdict
Biochemical,,,,,,
Progesterone,"Normal, lower side",raises it,1.04,3.49,4.52,Good
LDL Cholesterol,"Normal, upper side",lowers it,1.19,2.49,3.68,Good
Blood Glucose,High,lowers it,1.51,2.08,3.59,Good
HDL Cholesterol,"Normal, lower side",raises it,1.02,2.49,3.52,Good
C-Reactive Protein,High,lowers it,2.67,0.81,3.47,Good
Albumin/Globulin Ratio,"Normal, lower side",raises it,1.11,2.24,3.35,Good
Estrogen,"Normal, lower side",raises it,0.66,2.28,2.94,Good
Free Thyroxine (Free T4),"Normal, lower side",raises it,0.92,1.67,2.59,Good
Gamma-Glutamyl Transferase,High,raises it,1.45,-4.00,-2.55,Bad


In [11]:
bars(result.explain("Banana").head(12), "Total")

,Your result,This food,Direct,Through nutrients,Total,Verdict
Biochemical,,,,,,
Blood Glucose,High,raises it,-1.86,-3.43,-5.29,Bad
Hemoglobin A1c,"Normal, upper side",raises it,-0.97,-2.16,-3.14,Bad
Triglycerides,"Normal, upper side",raises it,-0.79,-1.97,-2.77,Bad
Thyroid-Stimulating Hormone (TSH),High,raises it,-1.47,-0.54,-2.00,Bad
Total Cholesterol,"Normal, upper side",raises it,-0.86,-0.78,-1.64,Bad
Uric Acid,High,raises it,0.00,-1.42,-1.42,Bad
Gamma-Glutamyl Transferase,High,raises it,0.00,-1.33,-1.33,Bad
Vitamin B6 (Pyridoxine),"Normal, upper side",raises it,-0.64,-0.48,-1.12,Bad
Aspartate Aminotransferase,"Normal, upper side",raises it,-0.68,-0.31,-0.99,Bad


## 8. Change a result and everything updates

Say the next visit shows high glucose and good vitamin D. Change those two numbers.
The lab table, the foods, the nutrients and the report all update.

In [12]:
after = result.update({"Blood Glucose": 180, "Vitamin D": 45})
after.labs().head(8)

,Result,Healthy range,Status,How far off
Biochemical,,,,
Blood Glucose,180 mg/dL,70 to 100,High,4.00
C-Reactive Protein,4 mg/L,0 to 3,High,1.67
Gamma-Glutamyl Transferase,43 U/L,5 to 36,High,1.45
Thyroid-Stimulating Hormone (TSH),4.4 mIU/L,0.4 to 4,High,1.22
Uric Acid,6.3 mg/dL,2.4 to 6,High,1.17
Iron,47.8 µg/dL,50 to 170,Low,-1.04
Total Cholesterol,198.3 mg/dL,125 to 200,"Normal, upper side",0.95
Progesterone,1.8 ng/mL,0.1 to 25,"Normal, lower side",-0.86


**Moved** shows how many places each food went up (positive) or down (negative).

In [13]:
after.compare(result)

,Rank,Was,Moved,Score
Food,,,,
Flax Seed,1,1,0,93.2
Chia Seed,2,2,0,91.9
Mint,3,4,1,88.1
Parsley,4,3,-1,87.9
Thyme,5,6,1,83.4
Oregano,6,10,4,83.1
Strawberry,7,8,1,82.0
Salmon,8,5,-3,81.9
Broccoli Sprouts,9,11,2,81.9


In [14]:
print(after.summary())

58 lab results, 6 outside the healthy range.
High: Blood Glucose 180 mg/dL (healthy 70 to 100); C-Reactive Protein 4 mg/L (healthy 0 to 3); Gamma-Glutamyl Transferase 43 U/L (healthy 5 to 36); Thyroid-Stimulating Hormone (TSH) 4.4 mIU/L (healthy 0.4 to 4); Uric Acid 6.3 mg/dL (healthy 2.4 to 6)
Low: Iron 47.8 µg/dL (healthy 50 to 170)

Eat more: Fiber, Omega-3 Fatty Acids, Antioxidants, Vitamin C, Polyphenols, Magnesium
Eat less: Trans Fatty Acids, Carbohydrates, Fructose, Glucose, Alcohol, Sucrose

Best 10 foods (score out of 100):
  1. Flax Seed  93.2  helps lower Blood Glucose and C-Reactive Protein
  2. Chia Seed  91.9  helps lower Blood Glucose and C-Reactive Protein
  3. Mint  88.1  helps lower Blood Glucose and C-Reactive Protein
  4. Parsley  87.9  helps lower Blood Glucose and C-Reactive Protein
  5. Thyme  83.4  helps lower Blood Glucose and C-Reactive Protein
  6. Oregano  83.1  helps lower Blood Glucose and C-Reactive Protein
  7. Strawberry  82  helps lower C-Reactive Prot

## 9. Under the hood: the matrix math

This part is for researchers. The whole recommendation is three matrix multiplications.

1. **deviation**: one number per blood test (58 numbers)
2. **drive** = deviation × importance × minus 1. A high test pushes down, a low test pushes up.
3. **direct** = drive @ food_coefficients. (58) @ (58 × 150) gives one number per food.
4. **need** = drive @ nutrient_coefficients. (58) @ (58 × 75) gives one number per nutrient.
5. **through** = food_nutrients @ need. (150 × 75) @ (75) gives one number per food.
6. **score** = direct scaled to 0..1, plus through scaled to 0..1, plus nutriscore.

Every coefficient is bias + weight. Bias is the direction: 1 means the food raises the test, minus 1 means it lowers it.
Weight is the strength. Both tables were written by hand in the research notebooks.

In [15]:
m = result.matrices()
pd.DataFrame({"shape": {name: str(x.shape) for name, x in m.items()}})

,shape
deviation,"(58,)"
drive,"(58,)"
food_coefficients,"(58, 150)"
nutrient_coefficients,"(58, 75)"
food_nutrients,"(150, 75)"
direct,"(150,)"
need,"(75,)"
through_nutrients,"(150,)"
parts,"(150, 3)"
score,"(150,)"


The deviation vector for this patient (tests sitting exactly in the middle are left out):

In [16]:
deviation = m["deviation"]
bars(deviation[deviation != 0].sort_values().to_frame(), "deviation")

,deviation
name,
Vitamin D,-1.11
Iron,-1.04
Progesterone,-0.86
HDL Cholesterol,-0.85
Free Thyroxine (Free T4),-0.77
Albumin/Globulin Ratio,-0.74
Apolipoprotein A1,-0.66
Estrogen,-0.60
Free Triiodothyronine (Free T3),-0.58


A corner of the food coefficient matrix. Positive means the food raises the test:

In [17]:
m["food_coefficients"].loc[["Blood Glucose", "LDL Cholesterol", "Vitamin D", "C-Reactive Protein"],
                           ["Flax Seed", "Salmon", "Banana", "Donuts"]]

name,Flax Seed,Salmon,Banana,Donuts
name,,,,
Blood Glucose,-1.3,-1.0,1.6,1.7
LDL Cholesterol,-1.4,-1.5,0.9,0.8
Vitamin D,0.0,2.0,0.0,0.0
C-Reactive Protein,-1.6,-1.7,0.0,1.7


Now do the math by hand and check that it gives the same scores as the package:

In [18]:
direct = m["drive"] @ m["food_coefficients"]       # 150 numbers, one per food
need = m["drive"] @ m["nutrient_coefficients"]     # 75 numbers, one per nutrient
through = m["food_nutrients"] @ need               # 150 numbers, one per food

scale = lambda x: (x - x.min()) / (x.max() - x.min())
score = scale(direct) + scale(through) + m["parts"]["nutriscore"]

print("Same as the package:", np.allclose(score, m["score"]))
score.sort_values(ascending=False).head(10).round(3)

Same as the package: True


name
Flax Seed       2.818
Chia Seed       2.789
Parsley         2.629
Mint            2.597
Salmon          2.536
Thyme           2.467
Sardines        2.466
Strawberry      2.464
Pumpkin Seed    2.463
Oregano         2.452
dtype: float64

The pull matrix (58 × 150) says how much each test moves each food. Here are the tests out of range,
against the 8 best and the 4 worst foods. Blue pushes the food up, red pushes it down.

In [19]:
off_range = result.labs().query("Status in ['High', 'Low']").index
columns = list(result.table.index[:8]) + list(result.table.index[-4:])
bars(m["pull"].loc[off_range, columns], *columns, axis=None)

name,Flax Seed,Chia Seed,Parsley,Mint,Salmon,Thyme,Sardines,Strawberry,Cakes,Muffins,Donuts,Brownies
Biochemical,,,,,,,,,,,,
C-Reactive Protein,3.47,3.31,4.89,5.44,1.58,4.11,0.80,2.58,-4.88,-5.40,-5.21,-6.00
Gamma-Glutamyl Transferase,-2.55,-2.33,-0.42,-0.75,-2.77,-2.58,-6.47,-0.42,-6.10,-7.13,-5.95,-7.80
Thyroid-Stimulating Hormone (TSH),-1.54,-1.97,-0.16,-0.45,-5.18,-1.21,-6.84,-0.51,-2.89,-3.69,-2.55,-4.07
Uric Acid,1.19,1.12,2.31,1.93,-0.41,2.93,-0.77,0.77,-7.28,-11.99,-6.34,-13.32
Blood Glucose,3.59,3.17,3.63,4.09,1.32,4.34,1.26,0.09,-7.52,-7.81,-6.86,-8.62
Vitamin D,2.23,2.46,1.75,0.65,3.81,2.12,5.12,0.08,0.40,0.52,0.33,0.76
Iron,0.44,0.21,0.58,-0.86,1.67,0.63,3.05,0.98,0.32,0.30,0.27,0.34


How much did this patient's results change the list? The research notebook measured how far each food
moved compared with a list sorted by nutriscore alone:

In [20]:
print(f"Each food moved {result.moved_places:.0f} places on average")

Each food moved 22 places on average


## 10. Which scaling to use

The live Biolabs backend (**legacy**) measures distance outside the healthy range in the test's own unit.
So a result just past the edge can look normal, and tests with big units count more.
The research notebook added 1 at the edge (**original**). **linear** measures every test in half ranges,
the same way inside and outside the range. This notebook uses linear.

Look at HbA1c 5.8 (the healthy max is 5.6). Legacy gives it 0.2, so the backend calls it normal.

In [21]:
border = {"Hemoglobin A1c": 5.8, "Thyroid-Stimulating Hormone (TSH)": 4.3,
          "Gamma-Glutamyl Transferase": 43, "Blood Glucose": 102}
table = pd.DataFrame({name: rec.deviations(border, "female", Tuning(scaling=name)).deviation
                      for name in ["legacy", "original", "linear"]}).round(2)
table.insert(0, "Result", pd.Series(border))
table

,Result,legacy,original,linear
biochemical,,,,
Hemoglobin A1c,5.8,0.2,1.2,1.25
Thyroid-Stimulating Hormone (TSH),4.3,0.3,1.3,1.17
Gamma-Glutamyl Transferase,43.0,7.0,8.0,1.45
Blood Glucose,102.0,2.0,3.0,1.13


`Tuning.legacy()` gives exactly the numbers the live backend gives today, if you need to match it.

## 11. Personal settings

An admin keeps one set of settings for everyone. Each person can get a small patch on top.
Only the patch needs to be saved for that person.

In [22]:
patch = {
    "importance": {"Blood Glucose": 2.0},   # glucose counts double for this person
    "ranges": {"Vitamin D": [30, 60]},      # their own target range
    "exclude_groups": ["meat"],             # vegetarian
    "exclude_foods": ["Shrimp"],            # allergy
}
personal = rec.recommend(patient, tuning=Tuning.recommended().update(patch))
personal.compare(result)

,Rank,Was,Moved,Score
Food,,,,
Flax Seed,1,1,0,93.1
Chia Seed,2,2,0,92.1
Parsley,3,3,0,87.2
Mint,4,4,0,86.1
Salmon,5,5,0,84.4
Sardines,6,7,1,82.7
Thyme,7,6,-1,82.6
Oregano,8,10,2,82.5
Pumpkin Seed,9,9,0,82.3


In [23]:
print(json.dumps(patch, indent=2))   # what you would save for this person

{
  "importance": {
    "Blood Glucose": 2.0
  },
  "ranges": {
    "Vitamin D": [
      30,
      60
    ]
  },
  "exclude_groups": [
    "meat"
  ],
  "exclude_foods": [
    "Shrimp"
  ]
}


## 12. Try the settings with sliders

Drag a slider and the table updates. It shows the new top 12 against the list from section 5.

In [24]:
import ipywidgets as w

style = {"description_width": "140px"}
slider = lambda label, value=1.0, top=3.0: w.FloatSlider(value=value, min=0, max=top, step=0.1, description=label,
                                                         style=style, layout=w.Layout(width="440px"))

def show(scaling, food_mix, nutrient_mix, nutriscore_mix, bias_scale, weight_scale, glucose, crp):
    tuning = Tuning(scaling=scaling, cap=4.0 if scaling == "linear" else None,
                    food_mix=food_mix, nutrient_mix=nutrient_mix, nutriscore_mix=nutriscore_mix,
                    bias_scale=bias_scale, weight_scale=weight_scale,
                    importance={"Blood Glucose": glucose, "C-Reactive Protein": crp})
    display(rec.recommend(patient, tuning=tuning).compare(result, n=12))

w.interact(show,
           scaling=w.ToggleButtons(options=["linear", "original", "legacy"], description="scaling", style=style),
           food_mix=slider("food (direct)"), nutrient_mix=slider("food (nutrients)"), nutriscore_mix=slider("nutriscore"),
           bias_scale=slider("bias"), weight_scale=slider("weight"),
           glucose=slider("glucose importance"), crp=slider("CRP importance"));

interactive(children=(ToggleButtons(description='scaling', options=('linear', 'original', 'legacy'), style=Tog…

## 13. Send it to your app

`to_json()` gives everything as JSON, ready for an API or a frontend.
Here is the start of it:

In [25]:
print(result.to_json()[:1200])

{
  "labs": [
    {
      "biochemical": "C-Reactive Protein",
      "result": "4 mg/L",
      "healthy_range": "0 to 3",
      "status": "High",
      "how_far_off": 1.67
    },
    {
      "biochemical": "Gamma-Glutamyl Transferase",
      "result": "43 U/L",
      "healthy_range": "5 to 36",
      "status": "High",
      "how_far_off": 1.45
    },
    {
      "biochemical": "Thyroid-Stimulating Hormone (TSH)",
      "result": "4.4 mIU/L",
      "healthy_range": "0.4 to 4",
      "status": "High",
      "how_far_off": 1.22
    },
    {
      "biochemical": "Uric Acid",
      "result": "6.3 mg/dL",
      "healthy_range": "2.4 to 6",
      "status": "High",
      "how_far_off": 1.17
    },
    {
      "biochemical": "Blood Glucose",
      "result": "102.4 mg/dL",
      "healthy_range": "70 to 100",
      "status": "High",
      "how_far_off": 1.16
    },
    {
      "biochemical": "Vitamin D",
      "result": "18.4 ng/mL",
      "healthy_range": "20 to 50",
      "status": "Low",
     

## 14. Use your own data

```python
from biofoodrec import FoodRecommender, ReferenceData

ref = ReferenceData.from_csv_dir("my_folder")                                 # all 7 files
ref = ReferenceData.from_csv_dir("my_folder", food_weight="new_weights.csv")  # swap one file
rec = FoodRecommender(ref)
```

Tables are matched by name, so the column order does not matter.
If a test or a food is missing, the error message names it.